# NB08 - Selective prediction (abstention) and Grad-CAM with sanity checks
* Rejection threshold per fold from the calibration set; risk-coverage curves, AURC, selective accuracy at 90 % / 80 % coverage.
* Grad-CAM on a **balanced** set of cases (TP, TN, FP, FN + random) - not only the most confident ones.
* Sanity checks: cascading weight randomisation, stability across independently trained models, descriptive overlap with CSF/GM/WM.

In [ ]:
# ── 1. Environment Detection (Google Colab vs Local Machine) ──────────────
import sys, os, subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── 2. Configure Paths ───────────────────────────────────────────────────
if IN_COLAB:
    # Google Colab paths (on your Google Drive)
    PROJECT  = '/content/drive/MyDrive/oasis_project'      # this unzipped project folder on Drive
    DATA_DIR = '/content/drive/MyDrive/OASIS1'             # folder with the 12 OASIS-1 disc archives (+ clinical csv)
    OUT_DIR  = '/content/drive/MyDrive/OASIS1_outputs'     # results are saved here (survives Colab disconnects)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{PROJECT}/requirements.txt'], check=False)
else:
    # Local machine paths (runs directly on your laptop)
    PROJECT  = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
    DATA_DIR = os.path.join(PROJECT, 'Dataset')
    OUT_DIR  = os.path.join(PROJECT, 'OASIS1_outputs')

# ── 3. install + load ─────────────────────────────────────────────────────
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
from oasis_ad.config import load_config, Paths
from oasis_ad import pipeline as PL, DISCLAIMER
cfg = load_config(os.path.join(PROJECT, 'configs', 'default.yaml'),
                  {'paths.data_dir': DATA_DIR, 'paths.out_dir': OUT_DIR})
paths = Paths(cfg)
import torch, pandas as pd
print(DISCLAIMER)
print('Environment:', 'Google Colab' if IN_COLAB else 'Local Laptop')
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (on Colab switch to Runtime > Change runtime type > T4 GPU)')

In [ ]:
MODEL = 'cnn3d'
res = PL.step8_selective_saliency(cfg, paths, MODEL)
import json; print(json.dumps(res['selective']['pooled_mean'], indent=2))
print(res['selective']['rejection_rule'])

In [ ]:
from IPython.display import Image, display
display(Image(f'{paths.selective}/{MODEL}/risk_coverage.png'))

In [ ]:
print('Randomisation check (correlation with the original map should DROP):')
res['randomization'].pivot_table(index='randomised_up_to', values='pearson_with_original').sort_values('pearson_with_original', ascending=False)

In [ ]:
print('Stability across models trained on different splits:'); display(res['stability'])
print('Share of high-activation voxels per tissue (descriptive):'); display(res['tissue_overlap'].groupby('kind').mean(numeric_only=True).round(3))

### Explanation stability under small input perturbations (correct vs wrong predictions)
Pearson correlation between the original Grad-CAM and the map after mild noise / 2-voxel shift / intensity change / smooth bias field.

In [ ]:
display(res['perturbation_stability'])

In [ ]:
import glob
for f in sorted(glob.glob(f'{paths.selective}/{MODEL}/gradcam_*.png'))[:8]: display(Image(f))